# 🚕 NYC Yellow Taxi - Data Engineering Pipeline
# Dataset: Yellow Taxi Trip Records - Março 2016

In [1]:
import pyarrow
import pandas as pd
import numpy as np
from pathlib import Path

## 1. Extract - Leitura dos Dados

In [2]:
df = pd.read_csv('../data/yellow_tripdata_2016-03.csv')
print(f"Total de registros: {len(df):,}")
print(f"Total de colunas: {len(df.columns)}")
df.head()

Total de registros: 12,210,952
Total de colunas: 19


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,pickup_longitude,pickup_latitude,RatecodeID,store_and_fwd_flag,dropoff_longitude,dropoff_latitude,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount
0,1,2016-03-01 00:00:00,2016-03-01 00:07:55,1,2.50,-73.976746,40.765152,1,N,-74.004265,40.746128,1,9.0,0.5,0.5,2.05,0.00,0.3,12.35
1,1,2016-03-01 00:00:00,2016-03-01 00:11:06,1,2.90,-73.983482,40.767925,1,N,-74.005943,40.733166,1,11.0,0.5,0.5,3.05,0.00,0.3,15.35
2,2,2016-03-01 00:00:00,2016-03-01 00:31:06,2,19.98,-73.782021,40.644810,1,N,-73.974541,40.675770,1,54.5,0.5,0.5,8.00,0.00,0.3,63.80
3,2,2016-03-01 00:00:00,2016-03-01 00:00:00,3,10.78,-73.863419,40.769814,1,N,-73.969650,40.757767,1,31.5,0.0,0.5,3.78,5.54,0.3,41.62
4,2,2016-03-01 00:00:00,2016-03-01 00:00:00,5,30.43,-73.971741,40.792183,3,N,-74.177170,40.695053,1,98.0,0.0,0.0,0.00,15.50,0.3,113.80


In [13]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 12210952 entries, 0 to 12210951
Data columns (total 21 columns):
 #   Column                 Dtype         
---  ------                 -----         
 0   VendorID               int64         
 1   tpep_pickup_datetime   str           
 2   tpep_dropoff_datetime  str           
 3   passenger_count        int64         
 4   trip_distance          float64       
 5   pickup_longitude       float64       
 6   pickup_latitude        float64       
 7   RatecodeID             int64         
 8   store_and_fwd_flag     str           
 9   dropoff_longitude      float64       
 10  dropoff_latitude       float64       
 11  payment_type           int64         
 12  fare_amount            float64       
 13  extra                  float64       
 14  mta_tax                float64       
 15  tip_amount             float64       
 16  tolls_amount           float64       
 17  improvement_surcharge  float64       
 18  total_amount           float64 

In [ ]:
df.describe()

## 2. Data Quality & Anomalias
Detectar e remover registros inválidos

In [3]:
# Converter colunas de data
df['pickup_datetime'] = pd.to_datetime(df['tpep_pickup_datetime'])
df['dropoff_datetime'] = pd.to_datetime(df['tpep_dropoff_datetime'])

# Verificar valores nulos
print("=== VALORES NULOS ===")
null_counts = df.isnull().sum()
print(null_counts[null_counts > 0] if null_counts.sum() > 0 else "Nenhum valor nulo encontrado")

=== VALORES NULOS ===
Nenhum valor nulo encontrado


In [15]:
# Detectar anomalias
total = len(df)

# Distância inválida
invalid_distance = (df['trip_distance'] <= 0) | (df['trip_distance'] > 100)

# Número de passageiros inválido
invalid_passengers = (df['passenger_count'] <= 0) | (df['passenger_count'] > 6)

# Tarifa negativa
invalid_fare = df['fare_amount'] < 0

# Duração inválida (pickup depois do dropoff)
invalid_duration = df['pickup_datetime'] > df['dropoff_datetime']

# Coordenadas fora de NYC (latitude ~40.4-41.0, longitude ~-74.3 a -73.7)
invalid_pickup_coords = (
    (df['pickup_latitude'] < 40.4) | (df['pickup_latitude'] > 41.0) |
    (df['pickup_longitude'] < -74.3) | (df['pickup_longitude'] > -73.7)
)
invalid_dropoff_coords = (
    (df['dropoff_latitude'] < 40.4) | (df['dropoff_latitude'] > 41.0) |
    (df['dropoff_longitude'] < -74.3) | (df['dropoff_longitude'] > -73.7)
)

print("=== ANOMALIAS DETECTADAS ===")
print(f"Distância inválida (<=0 ou >100mi): {invalid_distance.sum():,} ({invalid_distance.sum()/total*100:.2f}%)")
print(f"Passageiros inválidos (<=0 ou >6):  {invalid_passengers.sum():,} ({invalid_passengers.sum()/total*100:.7f}%)")
print(f"Tarifa negativa:                    {invalid_fare.sum():,} ({invalid_fare.sum()/total*100:.4f}%)")
print(f"Duração inválida:                   {invalid_duration.sum():,} ({invalid_duration.sum()/total*100:.7f}%)")
print(f"Coordenadas pickup fora de NYC:     {invalid_pickup_coords.sum():,} ({invalid_pickup_coords.sum()/total*100:.2f}%)")
print(f"Coordenadas dropoff fora de NYC:    {invalid_dropoff_coords.sum():,} ({invalid_dropoff_coords.sum()/total*100:.2f}%)")

=== ANOMALIAS DETECTADAS ===
Distância inválida (<=0 ou >100mi): 71,225 (0.58%)
Passageiros inválidos (<=0 ou >6):  678 (0.0055524%)
Tarifa negativa:                    4,581 (0.0375%)
Duração inválida:                   109 (0.0008926%)
Coordenadas pickup fora de NYC:     183,816 (1.51%)
Coordenadas dropoff fora de NYC:    178,754 (1.46%)


In [5]:
# Remover registros inválidos
invalid_mask = (
    invalid_distance |
    invalid_passengers |
    invalid_fare |
    invalid_duration |
    invalid_pickup_coords |
    invalid_dropoff_coords
)

df_clean = df[~invalid_mask].copy()
removed = total - len(df_clean)
print(f"Registros antes:   {total:,}")
print(f"Registros removidos: {removed:,} ({removed/total*100:.2f}%)")
print(f"Registros após:    {len(df_clean):,}")

Registros antes:   12,210,952
Registros removidos: 255,326 (2.09%)
Registros após:    11,955,626


## 3. Transform - Colunas Calculadas
Enriquecer o dataset com métricas derivadas

In [6]:
# Duração da corrida (minutos)
df_clean['trip_duration_min'] = (
    (df_clean['dropoff_datetime'] - df_clean['pickup_datetime'])
    .dt.total_seconds() / 60
)

# Velocidade média (milhas/hora)
hours = df_clean['trip_duration_min'] / 60
df_clean['trip_speed_mph'] = (
    df_clean['trip_distance'] / hours.replace(0, np.nan)
).fillna(0)

# Receita por milha
df_clean['revenue_per_mile'] = (
    df_clean['total_amount'] / df_clean['trip_distance'].replace(0, np.nan)
).fillna(0)

# Hora do dia
df_clean['hour_of_day'] = df_clean['pickup_datetime'].dt.hour

# Data (sem hora)
df_clean['date'] = df_clean['pickup_datetime'].dt.date

# Dia da semana
df_clean['day_of_week'] = df_clean['pickup_datetime'].dt.day_name()

# Gorjeta percentual
df_clean['tip_pct'] = (
    (df_clean['tip_amount'] / df_clean['fare_amount'].replace(0, np.nan)) * 100
).fillna(0)

print("=== NOVAS COLUNAS ===")
new_cols = ['trip_duration_min', 'trip_speed_mph', 'revenue_per_mile', 'hour_of_day', 'date', 'day_of_week', 'tip_pct']
for col in new_cols:
    print(f"  {col}")

print(f"\nTotal de colunas: {len(df_clean.columns)}")
df_clean[new_cols].head(10)

=== NOVAS COLUNAS ===
  trip_duration_min
  trip_speed_mph
  revenue_per_mile
  hour_of_day
  date
  day_of_week
  tip_pct

Total de colunas: 28


,trip_duration_min,trip_speed_mph,revenue_per_mile,hour_of_day,date,day_of_week,tip_pct
0,7.916667,18.947368,4.940000,0,2016-03-01,Tuesday,22.777778
1,11.100000,15.675676,5.293103,0,2016-03-01,Tuesday,27.727273
2,31.100000,38.546624,3.193193,0,2016-03-01,Tuesday,14.678899
3,0.000000,0.000000,3.860853,0,2016-03-01,Tuesday,12.000000
4,0.000000,0.000000,3.739731,0,2016-03-01,Tuesday,0.000000
5,0.000000,0.000000,5.128378,0,2016-03-01,Tuesday,21.531915
7,16.050000,23.177570,3.516129,0,2016-03-01,Tuesday,0.000000
8,4.983333,8.428094,12.571429,0,2016-03-01,Tuesday,36.363636
9,24.083333,17.887889,3.899721,0,2016-03-01,Tuesday,13.617021
10,2.033333,15.934426,9.814815,0,2016-03-01,Tuesday,0.000000


## 4. Agregações & Métricas
Sumários por hora, vendor, tipo de pagamento e zona

In [7]:
# Sumário por hora do dia
hourly = df_clean.groupby('hour_of_day').agg(
    total_trips=('hour_of_day', 'size'),
    avg_distance=('trip_distance', 'mean'),
    avg_fare=('fare_amount', 'mean'),
    avg_duration=('trip_duration_min', 'mean'),
    avg_speed=('trip_speed_mph', 'mean'),
    avg_tip_pct=('tip_pct', 'mean'),
).round(2)

print("=== SUMÁRIO POR HORA DO DIA ===")
hourly

=== SUMÁRIO POR HORA DO DIA ===


,total_trips,avg_distance,avg_fare,avg_duration,avg_speed,avg_tip_pct
hour_of_day,,,,,,
0,420453,3.46,13.28,15.46,17.49,15.82
1,301155,3.34,12.76,15.96,17.53,16.56
2,202960,3.30,12.52,14.55,18.02,14.07
3,164354,3.50,13.06,15.41,19.34,13.36
4,125356,4.28,15.04,15.39,22.23,12.02
5,122075,4.67,15.84,14.32,24.36,13.01
6,270668,3.60,13.10,12.87,18.34,13.67
7,460189,2.88,11.91,14.06,14.51,23.50
8,562177,2.61,12.05,16.23,11.80,14.87


In [8]:
# Sumário por vendor
vendor = df_clean.groupby('VendorID').agg(
    total_trips=('VendorID', 'size'),
    avg_distance=('trip_distance', 'mean'),
    avg_fare=('fare_amount', 'mean'),
    total_revenue=('total_amount', 'sum'),
    avg_tip_pct=('tip_pct', 'mean'),
    avg_speed=('trip_speed_mph', 'mean'),
).round(2)

print("=== SUMÁRIO POR VENDOR ===")
vendor

=== SUMÁRIO POR VENDOR ===


,total_trips,avg_distance,avg_fare,total_revenue,avg_tip_pct,avg_speed
VendorID,,,,,,
1,5557584,2.90,12.54,8.746789e+07,15.55,15.48
2,6398042,3.01,12.76,1.025844e+08,14.33,12.13


In [10]:
# Sumário por tipo de pagamento
payment_labels = {1: 'Credit Card', 2: 'Cash', 3: 'No Charge', 4: 'Dispute', 5: 'Unknown', 6: 'Voided'}
df_clean['payment_label'] = df_clean['payment_type'].map(payment_labels).fillna('Other')

payment = df_clean.groupby('payment_label').agg(
    total_trips=('payment_label', 'size'),
    avg_fare=('fare_amount', 'mean'),
    avg_tip=('tip_amount', 'mean'),
    avg_tip_pct=('tip_pct', 'mean'),
    total_revenue=('total_amount', 'sum'),
).round(2)

payment['pct_trips'] = ((payment['total_trips'] / payment['total_trips'].sum()) * 100).round(2)

print("=== SUMÁRIO POR TIPO DE PAGAMENTO ===")
payment

=== SUMÁRIO POR TIPO DE PAGAMENTO ===


,total_trips,avg_fare,avg_tip,avg_tip_pct,total_revenue,pct_trips
payment_label,,,,,,
Cash,3924767,11.70,0.00,0.00,5.122852e+07,32.83
Credit Card,7982953,13.11,2.67,22.31,1.380487e+08,66.77
Dispute,12807,14.27,0.00,0.03,2.046736e+05,0.11
No Charge,35099,14.31,0.00,0.02,5.703743e+05,0.29


In [9]:
# Sumário por dia da semana
weekday = df_clean.groupby('day_of_week').agg(
    total_trips=('day_of_week', 'size'),
    avg_distance=('trip_distance', 'mean'),
    avg_fare=('fare_amount', 'mean'),
    avg_duration=('trip_duration_min', 'mean'),
    avg_tip_pct=('tip_pct', 'mean'),
).round(2)

day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
weekday = weekday.reindex(day_order)

print("=== SUMÁRIO POR DIA DA SEMANA ===")
weekday

=== SUMÁRIO POR DIA DA SEMANA ===


,total_trips,avg_distance,avg_fare,avg_duration,avg_tip_pct
day_of_week,,,,,
Monday,1392304,3.03,12.64,15.27,14.65
Tuesday,1835690,2.89,12.56,15.58,14.96
Wednesday,1917604,2.90,12.75,16.08,15.42
Thursday,1993722,2.96,13.02,16.66,17.05
Friday,1661034,2.94,12.73,16.16,14.32
Saturday,1701429,2.82,12.06,15.42,13.31
Sunday,1453843,3.24,12.78,15.51,13.94


## 5. Load - Parquet
Salvar os dados limpos em formato Parquet

In [11]:
# Converter coluna date para string (pyarrow não serializa datetime.date)
df_clean['date'] = df_clean['date'].astype(str)

# Remover colunas originais de datetime (já temos pickup_datetime e dropoff_datetime)
df_clean = df_clean.drop(columns=['tpep_pickup_datetime', 'tpep_dropoff_datetime'])

output_path = Path('../data/output/yellow_taxi_2016-03.parquet')
output_path.parent.mkdir(parents=True, exist_ok=True)

df_clean.to_parquet(output_path, engine='pyarrow', index=False)

size_mb = output_path.stat().st_size / (1024 * 1024)
print(f"=== PARQUET ===")
print(f"Arquivo: {output_path}")
print(f"Tamanho: {size_mb:.1f} MB")

=== PARQUET ===
Arquivo: ../data/output/yellow_taxi_2016-03.parquet
Tamanho: 388.9 MB


In [12]:
# Validação: ler de volta o Parquet e conferir
df_parquet = pd.read_parquet(output_path)
print(f"=== VALIDAÇÃO ===")
print(f"Registros no Parquet: {len(df_parquet):,}")
print(f"Registros esperados:  {len(df_clean):,}")
print(f"Match: {'OK' if len(df_parquet) == len(df_clean) else 'ERRO'}")
print(f"\nColunas: {list(df_parquet.columns)}")
df_parquet.head()

=== VALIDAÇÃO ===
Registros no Parquet: 11,955,626
Registros esperados:  11,955,626
Match: OK

Colunas: ['VendorID', 'passenger_count', 'trip_distance', 'pickup_longitude', 'pickup_latitude', 'RatecodeID', 'store_and_fwd_flag', 'dropoff_longitude', 'dropoff_latitude', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'pickup_datetime', 'dropoff_datetime', 'trip_duration_min', 'trip_speed_mph', 'revenue_per_mile', 'hour_of_day', 'date', 'day_of_week', 'tip_pct', 'payment_label']


,VendorID,passenger_count,trip_distance,pickup_longitude,pickup_latitude,RatecodeID,store_and_fwd_flag,dropoff_longitude,dropoff_latitude,payment_type,...,pickup_datetime,dropoff_datetime,trip_duration_min,trip_speed_mph,revenue_per_mile,hour_of_day,date,day_of_week,tip_pct,payment_label
0,1,1,2.50,-73.976746,40.765152,1,N,-74.004265,40.746128,1,...,2016-03-01,2016-03-01 00:07:55,7.916667,18.947368,4.940000,0,2016-03-01,Tuesday,22.777778,Credit Card
1,1,1,2.90,-73.983482,40.767925,1,N,-74.005943,40.733166,1,...,2016-03-01,2016-03-01 00:11:06,11.100000,15.675676,5.293103,0,2016-03-01,Tuesday,27.727273,Credit Card
2,2,2,19.98,-73.782021,40.644810,1,N,-73.974541,40.675770,1,...,2016-03-01,2016-03-01 00:31:06,31.100000,38.546624,3.193193,0,2016-03-01,Tuesday,14.678899,Credit Card
3,2,3,10.78,-73.863419,40.769814,1,N,-73.969650,40.757767,1,...,2016-03-01,2016-03-01 00:00:00,0.000000,0.000000,3.860853,0,2016-03-01,Tuesday,12.000000,Credit Card
4,2,5,30.43,-73.971741,40.792183,3,N,-74.177170,40.695053,1,...,2016-03-01,2016-03-01 00:00:00,0.000000,0.000000,3.739731,0,2016-03-01,Tuesday,0.000000,Credit Card
